# TmY16 / Tm33 は見落とされた側抑制媒介か — 2点の確認

媒介候補スキャン(`src/lateral` 基盤、別ノート)で、**TmY16・Tm33** が「抑制出力優勢・数カラムを近傍
プール・段内抑制」という局所側抑制媒介の署名を示した。TmY/Tm は *feedforward relay* として扱われがち
だが NT 的には不均一(グルタミン酸性・GABA 性も存在)で、これら特定型の *近傍プール+段内抑制という
側抑制的役割* は報告が薄い。候補を主張に格上げするため2点を確認する:

1. **符号は本物か(NT 誤予測でないか)** — `nt_type` は主に ML 推定。確率的符号
   (`gaba_avg+glut_avg` vs `ach_avg`)で抑制性がロバストか、既知の興奮性 (Tm3) / 抑制性 (Mi4) と比較。
2. **実際に Δcolumn surround を作るか** — 媒介特異的 disynaptic kernel(exc→M→標的)を標的の home column
   基準で再構成。**ヘックス格子は距離 d に列が ~6d 個あり、生の合算カーネルは大きい d に偏る**ため、
   リング占有 N(d) で正規化した *per-column* プロファイルで「興奮中心 vs 抑制 surround」の形を比べる。

留保: 連結体由来の構造的推定であり、機能確定には physiology が必要。GLUT の機能的符号は受容体依存。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                  if (p / "src").is_dir() and (p / "pyproject.toml").exists()), Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))

from src.config import DATA_DIR
from src.data import FlyWireDataManager
from src.lateral import (
    STAGE_RANK,
    ColumnGeometry,
    LateralInhibitionCriteria,
    add_sign,
    assign_stage_from_manager,
    classify_inhibition,
    hex_distance,
    load_column_assignment,
    pq_hemi_maps,
    rms_radius,
)

HEMI = "right"
MEDS = ["TmY16", "Tm33"]              # candidate mediators
LATERAL_LABELS = {"direct_lateral", "wide_field_lateral"}
MAXD = 8

m = FlyWireDataManager()
conn = add_sign(m.optic_lobe_connections_df.copy())
ca = load_column_assignment(DATA_DIR)
geom = ColumnGeometry.from_assignment(ca)
P, Q, HM = pq_hemi_maps(ca)
stage = assign_stage_from_manager(m)
nd = m.optic_lobe_neurons_df
rank = stage.assign(r=stage["stage"].map(STAGE_RANK)).set_index("root_id")["r"].to_dict()
BD = geom.cells.dropna(subset=["region"]).drop_duplicates("root_id").set_index("root_id")["boundary_distance"].to_dict()
print(f"edges={len(conn):,}")

## 1. 符号のロバスト性 — NT 誤予測のアーティファクトでないか

各細胞の確率的符号 `P(inh)=gaba_avg+glut_avg`、`P(exc)=ach_avg` を比較する。出力シナプス数で重み付けた
抑制確率質量も見る。既知の興奮性 **Tm3** と既知 GABA 性 **Mi4** を対照に置く。

In [ ]:
osf = conn.groupby("pre_root_id")["syn_count"].sum()   # output synapses per cell (weighting)
rows = []
for t in MEDS + ["Tm3", "Mi4"]:
    s = nd[nd["primary_type"] == t]
    p_inh = s["gaba_avg"].fillna(0) + s["glut_avg"].fillna(0)
    p_exc = s["ach_avg"].fillna(0)
    w = s["root_id"].map(osf).fillna(0).to_numpy()
    wsum = w.sum() if w.sum() > 0 else 1
    rows.append(dict(type=t, n=len(s),
                     med_P_inh=round(float(p_inh.median()), 2),
                     med_P_exc=round(float(p_exc.median()), 2),
                     frac_cells_inhibitory=round(float((p_inh > p_exc).mean()), 2),
                     outsyn_weighted_P_inh=round(float((p_inh.to_numpy() * w).sum() / wsum), 2)))
sign_tbl = pd.DataFrame(rows)
display(sign_tbl)

**読み:** TmY16/Tm33 は P(inh)≈0.86–0.88・P(exc)≈0.06–0.09 で、既知 GABA 性の Mi4(0.93)と同等に
ロバストな抑制性。興奮性対照 Tm3(P(exc)=0.70)とは明確に分離する。確率的符号にしても抑制性の結論は
変わらず、「実は興奮性」説はほぼ否定される。残る不確実性は GABA か GLUT か、および GLUT の機能的符号
(受容体依存)であって、抑制 vs 興奮の別ではない。

## 2. 実際に Δcolumn surround を作るか — 媒介特異的 disynaptic kernel

motif **exc(X) → M(J) → 標的(T)**。標的 T の各内部細胞 t(boundary_distance≥3、エッジ切断回避)について、
M 由来の抑制が *どの列から* プールされた興奮に由来するかを、t の home column 基準の Δcolumn で集計する
(重み `w(M→t)·正規化 w(src→J)`)。

**重要:** 生の合算カーネル `f(d)` は距離 d の列数 N(d)~6d に比例して大きい d へ偏るので、ピーク位置は
形状の指標にならない。下で **リング占有 N(d) で正規化した per-column プロファイル `g(d)=f(d)/N(d)`** を
主指標とし、興奮中心と比較する。`rms_radius` の σ は未正規化(リポジトリ規約)で、同一 binning 同士の
*相対*比較にのみ用いる(絶対「半径」としては大きい d を過大評価する)。

In [ ]:
# ring occupancy N(d): mean #lattice columns at hex distance d from interior home columns
_cols = geom.columns[geom.columns.hemisphere == HEMI]
_pts = _cols[["p", "q"]].to_numpy()
_interior = _cols[_cols.boundary_distance >= 3][["p", "q"]].to_numpy()
Nd = np.zeros(MAXD + 1)
for p0, q0 in _interior:
    dd = hex_distance(_pts[:, 0] - p0, _pts[:, 1] - q0, integer=True)
    for d in range(MAXD + 1):
        Nd[d] += int((dd == d).sum())
Nd /= len(_interior)
print("ring occupancy N(d):", np.round(Nd, 1))

cl = classify_inhibition(conn, stage, col_assign=ca, criteria=LateralInhibitionCriteria(min_syn=1))
exc = conn[conn["sign"] == "exc"][["pre_root_id", "post_root_id", "syn_count"]].copy()
exc["sp"] = exc.pre_root_id.map(P); exc["sq"] = exc.pre_root_id.map(Q); exc["sh"] = exc.pre_root_id.map(HM)
exc = exc.dropna(subset=["sp", "sq"]); exc = exc[exc.sh == HEMI]
exc["pr"] = exc.pre_root_id.map(rank); exc["por"] = exc.post_root_id.map(rank)
col_types = set(ca[ca.hemisphere == HEMI]["type"].value_counts().loc[lambda s: s >= 50].index)


def raw_kernel(dfp):
    """Σ weight by integer Δcolumn between source column and post cell's home column."""
    d = hex_distance(dfp["sp"] - dfp["post_root_id"].map(P),
                     dfp["sq"] - dfp["post_root_id"].map(Q), integer=True)
    return dfp.assign(d=d).groupby("d")["w"].sum().reindex(range(MAXD + 1), fill_value=0).astype(float)


def percol_home_norm(f):
    """Per-column profile g(d)=f(d)/N(d), normalised to the home column (d=0)."""
    g = f / Nd
    return g / g.iloc[0] if g.iloc[0] > 0 else g


fig, axes = plt.subplots(len(MEDS), 3, figsize=(14, 4.4 * len(MEDS)))
axes = np.atleast_2d(axes)
summary = []
for mi, M in enumerate(MEDS):
    medge = conn[(conn.pre_primary_type == M) & (conn["sign"] == "inh")]
    tgt_rank = medge.groupby("post_primary_type")["syn_count"].sum().sort_values(ascending=False)
    targets = [t for t in tgt_rank.index if t in col_types][:3]
    J = set(nd.loc[nd.primary_type == M, "root_id"])
    jin = exc[exc.post_root_id.isin(J)].copy()
    jin["w2n"] = jin["syn_count"] / jin.groupby("post_root_id")["syn_count"].transform("sum")
    jin = jin.rename(columns={"post_root_id": "J"})[["J", "sp", "sq", "w2n"]]
    for ti, T in enumerate(targets):
        tcells = ca[(ca.type == T) & (ca.hemisphere == HEMI)].drop_duplicates("root_id")
        cids = set(tcells[tcells.root_id.map(BD).fillna(-1) >= 3].root_id)   # interior cells
        cen = exc[(exc.post_root_id.isin(cids)) & (exc.pr <= exc.por)][
            ["post_root_id", "sp", "sq", "syn_count"]].rename(columns={"syn_count": "w"})
        me = medge[medge.post_root_id.isin(cids)][["pre_root_id", "post_root_id", "syn_count"]] \
            .rename(columns={"pre_root_id": "J", "syn_count": "w1"})
        mg = me.merge(jin, on="J", how="inner"); mg["w"] = mg["w1"] * mg["w2n"]
        fc = raw_kernel(cen); fs = raw_kernel(mg[["post_root_id", "sp", "sq", "w"]])
        gc = percol_home_norm(fc); gs = percol_home_norm(fs)             # per-column, home-normalised
        sig_c, _ = rms_radius(fc); sig_s, _ = rms_radius(fs)             # un-normalised σ (relative only)
        tot_lat = cl[(cl.post_root_id.isin(cids)) & (cl.label.isin(LATERAL_LABELS))]["syn_count"].sum()
        m_lat = cl[(cl.post_root_id.isin(cids)) & (cl.label.isin(LATERAL_LABELS))
                   & (cl.pre_primary_type == M)]["syn_count"].sum()
        share = round(float(m_lat) / tot_lat, 3) if tot_lat > 0 else 0.0
        summary.append(dict(mediator=M, target=T, n_tcells=len(cids),
                            center_rms_unnorm=round(sig_c, 2), surround_rms_unnorm=round(sig_s, 2),
                            center_percol_d1=round(float(gc.iloc[1]), 2),     # excitation: ~0 already at d=1
                            surround_percol_d2=round(float(gs.iloc[2]), 2),   # inhibition: still high at d=2
                            surround_percol_d4=round(float(gs.iloc[4]), 2),
                            M_share_of_T_lateral=share))
        ax = axes[mi, ti]; r = np.arange(MAXD + 1)
        ax.plot(r, gc.values, "-o", color="tab:red", label=f"exc center (per-col)")
        ax.plot(r, gs.values, "-^", color="tab:blue", label=f"{M} surround (per-col)")
        ax.set(title=f"{M} → {T}   (M share {share:.0%})", xlabel="Δcolumn",
               ylabel="per-column weight (home=1)")
        ax.legend(fontsize=7); ax.grid(alpha=0.3)
    for ti in range(len(targets), 3):
        axes[mi, ti].axis("off")
plt.tight_layout()
display(pd.DataFrame(summary))

**読み(リング正規化後の正しい形状):** per-column の重み `g(d)`(home=1 に正規化)で見ると、抑制 surround
は **home(d=0)で最大・単調減衰**であり、隣接列にピークを持つドーナツ型ではない(生カーネルの d=2 ピークは
列数 N(d)~6d による集計アーティファクト)。**ただし減衰が緩慢**で、d=2 でも ~0.6、d=4 でも ~0.26 残る。
一方 **興奮中心は d=1 で既に ~0.05 に落ちる**。つまり「狭い興奮中心 vs 広い抑制」という DoG 的な空間拮抗が
成立しており、TmY16/Tm33 は実際に近傍をプールして標的へ広く抑制を返す。

**規模は補助的**: TmY16 は Mi4/Mi9 の側抑制の **11–13%**、Tm33 は Tm20 へ ~10% を担うにとどまり
(主体は正典の Dm/Pm amacrine)、それ以外の標的では <1%。主役の surround 生成器ではない。

## 結論

- **符号:** TmY16/Tm33 は確率的にも頑健に抑制性(Mi4 と同等、Tm3 と明確に分離)。NT 誤予測のアーティ
  ファクトではない。
- **幾何:** per-column で home 最大・緩慢減衰の **広い抑制 surround**(d=4 でも ~0.26)を、点状の興奮中心
  (d=1 で ~0.05)に重ねる — DoG 的な中心-周辺拮抗が成立(隣接列ピークの「ドーナツ」ではない)。
- **規模:** 補助的。最も強い具体的主張は **「TmY16 は Mi4・Mi9 への過小評価された
  グルタミン酸/GABA 性側抑制媒介で、広い抑制 surround を作り、それらの側抑制の 1 割強を占める」**。

残る限界: 構造的推定(機能確定は physiology)、寄与は小さい、GLUT の機能的符号は受容体依存。
次の詰め: 左半球・両半球一致、確率的符号での感度解析、TmY16→Mi4/Mi9 を主役にした深掘り。